# Singleton Creational desing pattern

### A singleton is a desing patter whose goal is:
- Make sure a class has exactly one "shared" instance
- Provide a single glocal access point to that instance

This of it like: "There should be one and only one manager object for this thing, and everyone uses the same one"

### Start from zero: what problem is htis trying to solve?
If real programs you often have "shared resources" or "shared state"
- **Configuration** (loaded once from env / YAML / secrets)
- **Logger** (one setup, same formattign/handles everywhere)
- **Database connection pool** (expensive to create; must be shared)
- **Cache client** (Redis client)
- **Telemetry / metrics** (one exporter)
- **Feature flags** or "app settings" objects

If you let every part of your code do:
- "I need config -> create config"
- "I need logger -> create logger"
- "I need DB pool -> create pool"

...you can get:

- **Duplicate expensive initialization** (slow startup, extra memory)
- **Inconsistent behaviour** (different modules configured differently)
- **Too many connections** (DB/Redis)
- **Hard-to-debug bugs** (two caches, two loggers, two config objects)


### The core idea in plain words

A Singleton class:
- **hides** how you create it (you can't freely call Class() anymore)
- **stores** a single instance inside the class
- **returns the same instance** to everyone who asks

So instead of:
* "Every time you need it, construct a new one"
 you get:
* "Every time you need it, fetch the same one

### **When to use Singleton (good use-cases)**

Use it when **ALL** are true:

1) **You truly want one "source of truth"**
- Example: global app configuration object. You don't want 5 configs.

2) **Creating it is expensive or should happen once**
- Example: DB pool, Redis client, model weights loading, SDK initialization

3) **Multiple instances woul cause real problems**
- Example: multiple loggers attaching multiple handlers -> duplicate logs.

4) **The instance if "stateless" or carefully controleld state**
- Singletons with lots of mutable state can become nightmare debugging

### **When NOT to use it (common traps)**

1) **You wan easy access everywhere**
- Thats tempting, but it becomes hidden global state

2) **Testing becomes painful**
- Singletosn can "stack around" acrosss tests and leak state

3) **You code becomes tightly coupled**
- Everything depends on Singleton.get_instance() rather than explicit dependecies

4) **Concurenccy / async initialization issues**
- If multiple threads initilize a once, you may accidentally create 2 instances


In many production Python codebases, people prefer Dependency Injection (DI) or module-level singletons (simple and explicit) 
rather that sttrict GoF-style Singleton classes



### **What Singleton really gives you (and what it doesn't)**

**It guarantees:**
- One instance (if implmemnted correcly)
- Centralized access

**It does NOT automatically guarantee**:
- Thread safety
- Safe mutation
- Good arhitecture
- Easy testing

Those are on you.


### **Singleton in real systems: what it looks like in practice**

**Example 1: One DB engine/ session factory**
- You want one pool per process
- You don't want every repository building its own engine

Singleton-ish approach:
- create engine once, reuse

**Example 2: Settings object**
- read env once
- parse YAML once
- store normalized config


**Example 3: LLM client wrapper**
- one client with retry policy
- one telemtery integration
- one rate limiter shared accross app

### How to decide: "Sholud i use Singleton here?"

**Yes if:**
- Two instances would break correctness or waster resources badly
- You need exactly oe global coordinator
- Lifetime matches teh app lifetime (startup -> shutdown)
- You can keep it mostly immutable after init

**No if:**
- It's just for convenience
- You expect multiple configurations (dev/test/prod)
- You want multiple clienst with different credentials/endpoints
- You need clen tests without global state

### **Testint considereations (the hidden pain point)**
Singletons are hard in tests because:
- State persists between tests
- Order of tests changes behaviour

If you use Singleton, add one of:
- a reset_for_tests() method
- ability to inject dependencies (preffered)
- avoid mutation after init

Example idea:
- AppConfig.load(...) once at startup, then tread it read-only

### **Common mistakes**

1. **Using Singleton for everything**
- This turns you app into *global state soup**

2. **Storing mutable business state inside**
- E.g "current user", "current requests" etc. Thats' usually wrong.

3. **Not thread-safe initialization**
- Two threads call at the same time -> two instances.

4. **Hidden dependencies**
- A function that silently uses GlocalLogger() is harser to reasong about htat one that recies logger

# Singleton in Python

### **Production use cases**
In production Python services (FastAPI/Flask/microservices), singletons are typically:

- Settings / config (env + YAML + secrets)
- Logger (handlers configured once)
- DB engine / connection pool (SQLAlchemy engine, async engine)
- HTTP client (requests.Session, https.AsyncClient)
- Redis client
- LLM client / SDK client with retries + timeouts
- Metrics/telemetry exporter

You alsom never need "Singleton because pattern", you need one shared instance per process.


# 1) Module-level singleton (MOST Pythonic, most used)

### **When to use**
- You want one object per process
- Simple lifecycle (created at import or via init function)
- You want easy usage: from x import client

### **Why it works**

Python caches imported modules in sys.modules. Importing again doesn't re-run the module body; you get the same module object -> same variables.

### Pattern A: eager creation at import

In [6]:
# Mocked create_engine, to simmulate connection to db
def create_engine():
    print("...")
    
engine = create_engine()

...


### Pros
- Dead simple
- Very common

### Cons
- Creating at import-time can be bad if:
    - config ins't loaded yet
    - you want laze init
    - you want different settings per test

### Pattern B: laze initialization with a getter

In [19]:
from typing import Optional
import redis

_client: Optional[redis.Redis] = None

def get_redis() -> redis.Redis:
    global _client
    if _client is None:
        _client = redis.Redis(host="localhost", port=6379, decode_responses=True)
    return _client


client_a = get_redis()
client_b = get_redis()
client_c = get_redis()

print(client_a is client_b)
print(client_a is client_c)
print(client_b is client_c)

True
True
True


### Problem:
If two threads call get_redis() simultaneously, you could create two clients. Usually not fatal, but if you care, add a lock (shown later)

### Pattern C: "initialize once" + "get"

Good for config / logging

In [26]:
from dataclasses import dataclass
from typing import Optional

@dataclass(frozen=True)
class Settings:
    env: str
    timeout: int
    
_settings: Optional[Settings] = None

def init_settings(*, env: str, timeout: int) -> None:
    global _settings
    if _settings is not None:
        raise RuntimeError("Settigns already initialized")
    _settings = Settings(env=env, timeout=timeout)
    
def get_settings() -> Settings:
    if _settings is None:
        raise RuntimeError("Settings not initialized")
    return _settings

init_settings(env="...", timeout=0)
get_settings()


Settings(env='...', timeout=0)

### Pros
- Prevents accidental "re-init"
- Super testable (you can reset in fixture)

# 2) Function caching singleton: functools.lru_cache / cache (super practical)

### **When to use**
- You want laze singleton
- You want thread-safe-ish behaviour "good enought"
- You want very compact code

In [28]:
from functools import lru_cache
import httpx

@lru_cache(maxsize=1)
def get_http_client() -> httpx.Client:
    return httpx.Client(timeout=10.0)

client_a = get_http_client()
client_b = get_http_client()

print(client_a is client_b)

True


### **Pros**
- Minimal boilerplace
- Lazy
- Great for getting objects, SDK clients, "heavy init"

### **Cons**
- Harder to "reset" unsell you call get_http_client.cache_clear()
- If you need teardown/close (httpx client), you must handle shutdown explicitly

**Reset example for tests**

In [29]:
def teardown():
    get_http_client.cache_clear()

# 3) Singleton class via __new__ (classic, but has Python gotchas)

**When to use**
- You really want "calling the class gies same instance"
- You want a strict Singleton class pattern (Gof style)

**Correct thread-safe version (with init guard)**

In [32]:
import threading

class Singleton:
    _instance = None
    _lock = threading.Lock()
    
    def __new__(cls, *args, **kwargs):
        if cls._instance is None:
            with cls._lock:
                if cls._instance is None:
                    cls._instance = super().__new__(cls)
        return cls._instance
    
class Config(Singleton):
    def __init__(self, env: str = "prod"):
        if getattr(self, "_initialized", False):
            return
        self._initialized = True
        self.env = env
        
config_a = Config()
config_b = Config()

print(config_a is config_b)

True


### Key gotcha

Even if __new__ returns the same object, __init__ is callsed every time you do Config(). That's why _initialized guard exists.

**Pros**
- Works
- Familiar to people who know GoF patterns

**Cons**
- More complex that need for python
- Testing reset is awkward (you must set Config._istance = None)

# 4) Metaclass Singleton (cleanes "formal" OOP singleton)

**When to use**
- You want multiple singleton classes with one reusable mechanism
- You want to avoid the __init__ multiple call issues cleanly

In [34]:
import threading

class SingletonMeta(type):
    _instances = {}
    _lock = threading.Lock()
    
    def __call__(cls, *args, **kwargs):
        if cls not in cls._instances:
            with cls._lock:
                if cls not in cls._instances:
                    instance = super().__call__(*args, **kwargs)
                    cls._instances[cls] = instance
        return cls._instances[cls]
    
class Logger(metaclass=SingletonMeta):
    def __init__(self, level="INFO"):
        self.level = level

**Pros**
- Elegant
- __init__ runs once (because __call__ only constructs once)

**Cons**
- Metaclasses confuse many teams
- Rest/testint rquires clearint SingletonMeta._instances

# 5) Borg patter (a.k.a "shared state", not identity)

### What it is

All instancs are differenct object, but the sahre the same internal state (__dict__)

In [35]:
class Borg:
    _shared_state = {}
    
    def __init__(self):
        self.__dict__ = self._shared_state
        
class Cache(Borg):
    def __init__(self):
        super().__init__()
        if not hasattr(self, "data"):
            self.data = {}

**What it's used**
Rarely. Sometimes for "shared config state" across instances.

**Pros**:
- No "one instance" requirement; just shared state

**Cons**:
- Very easy to create suprosing bugs
- Hardre to reason about than module-level

# 6) Choosing the right implmeenttion for real cases

- **Settings / config**
    - **Best**: @lru_cache or module-level get_settings()

- **Logger**
    - **Best**: module-leel logger = logging.getLogger(...) + configure once

- **DB engine/pool (SQLAlchemy)**
    - **Best**: module-level engine or @lru_cache getter
    - **For FastAPI**: create in lifespan

- **Async clients (https, async db)**
    - **Best**: FastAPI lifespan-anagerd app.state.*

- **SDK clienst (OpenAI / Azure / Bedrock / etc.)**
    - **Best**: @lru_cache(maxsize=1) getter of lifspan (if needs close)


### Template you can reuse

In [37]:
import threading
from typing import Optional

class ResourceManager:
    _lock = threading.Lock()
    _instance: Optional["ResourceManager"] = None
    
    def __new__(cls):
        if cls._instance is None:
            with cls._lock:
                if cls._instance is None:
                    cls._instance = super().__new__(cls)
                    cls._instance._initialized = False
        return cls._instance
    
    def init(self, *, db_url: str):
        if self._initialized:
            return
        self._initialized = True
        self.db_url = db_url
        
    def close(self):
        self._initialized = False
        
    @classmethod
    def reset_for_tests(cls):
        with cls._lock:
            if cls._instance is not None:
                cls._instance.close()
            cls._instance = None